In [7]:
import os
import numpy as np
import rasterio
from collections import defaultdict
from tqdm import tqdm

In [8]:
def gaussian_weights(n, sigma=1.0):
    """
    Generate Gaussian weights for a given number of elements.
    
    Args:
        n (int): Number of elements.
        sigma (float): Standard deviation of the Gaussian distribution.
        
    Returns:
        list: List of weights.
    """
    position = np.arange(n)
    center = (n - 1) / 2
    weights = np.exp(-(position - center)**2 / (2 * sigma**2))
    return weights / weights.sum()

In [ ]:
import os
import numpy as np
import rasterio
from collections import defaultdict
from tqdm import tqdm

# ==== CONFIG ====
input_dir = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/clean_1D_U_Net/deep_disturbance/Multikernel_Model/predictions'
output_file = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/clean_1D_U_Net/deep_disturbance/Multikernel_Model/final_yearly_predictions.tif'
threshold = 0.5  # threshold after weighted averaging
# ================

# Collect predictions
yearly_preds = defaultdict(list)

# Collect metadata from first file
first_file = sorted(os.listdir(input_dir))[0]
with rasterio.open(os.path.join(input_dir, first_file)) as src:
    meta = src.meta.copy()
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs
    dtype = src.dtypes[0]

# Traverse all files
for fname in tqdm(sorted(os.listdir(input_dir))):
    if not fname.endswith('.tif'):
        continue
    path = os.path.join(input_dir, fname)
    
    try:
        with rasterio.open(path) as src:
            for i in range(1, src.count + 1):
                try:
                    band_data = np.zeros((height, width), dtype=np.float32)
                    for ji, window in src.block_windows(i):
                        block_data = src.read(i, window=window)
                        band_data[window.row_off:window.row_off + window.height,
                                  window.col_off:window.col_off + window.width] = block_data
                    
                    band_year = int(src.descriptions[i - 1].split()[-1])
                    
                    if not np.isfinite(band_data).all():
                        print(f"Warning: non-finite values in {fname}, band {i}, replaced with 0")
                        band_data = np.nan_to_num(band_data, nan=0, posinf=0, neginf=0)
                    
                    yearly_preds[band_year].append(band_data)
                    
                except Exception as e:
                    print(f"Error reading band {i} from {fname}: {e}")
                    yearly_preds[band_year].append(np.zeros((height, width), dtype=np.float32))
                    
    except Exception as e:
        print(f"Error opening file {fname}: {e}")
        continue

# Aggregate with Gaussian weighting
sorted_years = sorted(yearly_preds.keys())
aggregated_predictions = []

for year in sorted_years:
    stack = np.stack(yearly_preds[year], axis=0)  # shape: [n_preds, H, W]
    
    if stack.shape[0] <= 2:
        print(f"Year {year}: only {stack.shape[0]} predictions -- taking their mean directly")
        vote = (np.mean(stack, axis=0) > threshold).astype(np.uint8)
        aggregated_predictions.append(vote)
        continue
    
    # Drop first & last predictions
    stack = stack[1:-1, :, :]
    
    # Gaussian weights
    weights = gaussian_weights(stack.shape[0], sigma=sigma)
    
    # Weighted mean
    weighted_mean = np.tensordot(weights, stack, axes=(0, 0))
    
    # Threshold to binary mask
    vote = (weighted_mean > threshold).astype(np.uint8)
    
    disturbed_pixels = np.sum(vote)
    total_pixels = vote.size
    print(f"\nYear {year}:")
    print(f"Predictions after dropping: {stack.shape[0]}")
    print(f"Disturbed pixels: {disturbed_pixels} ({(disturbed_pixels / total_pixels) * 100:.2f}%)")
    
    aggregated_predictions.append(vote)


# Save final raster
meta.update({
    "count": len(sorted_years),
    "dtype": rasterio.uint8,
    "compress": "lzw",
    "driver": "GTiff",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "interleave": "band",
    "bigtiff": "YES",
    "predictor": 2,
    "zlevel": 1
})

with rasterio.open(output_file, "w", **meta) as dst:
    for i, year in enumerate(sorted_years):
        arr = aggregated_predictions[i]
        if not np.isfinite(arr).all():
            arr[~np.isfinite(arr)] = 0
        if arr.dtype != np.uint8:
            arr = arr.astype(np.uint8)
        dst.write(arr, i + 1)
        dst.set_band_description(i + 1, f"Final prediction for year {year}")

print(f"\nSuccessfully saved Gaussian-weighted yearly predictions to {output_file}")


100%|██████████| 31/31 [00:23<00:00,  1.29it/s]


Year 1985: only 1 predictions → taking their mean directly
Year 1986: only 2 predictions → taking their mean directly

Year 1987:
Predictions after dropping: 1
Disturbed pixels: 1762234 (7.05%)

Year 1988:
Predictions after dropping: 2
Disturbed pixels: 532621 (2.13%)

Year 1989:
Predictions after dropping: 3
Disturbed pixels: 2587948 (10.35%)

Year 1990:
Predictions after dropping: 4
Disturbed pixels: 2081595 (8.33%)

Year 1991:
Predictions after dropping: 5
Disturbed pixels: 1719130 (6.88%)

Year 1992:
Predictions after dropping: 6
Disturbed pixels: 1695044 (6.78%)

Year 1993:
Predictions after dropping: 6
Disturbed pixels: 1144810 (4.58%)

Year 1994:
Predictions after dropping: 6
Disturbed pixels: 1804628 (7.22%)

Year 1995:
Predictions after dropping: 6
Disturbed pixels: 907596 (3.63%)

Year 1996:
Predictions after dropping: 6
Disturbed pixels: 981588 (3.93%)

Year 1997:
Predictions after dropping: 6
Disturbed pixels: 2031093 (8.12%)

Year 1998:
Predictions after dropping: 6
Distur